# Schema Evolution

Monday morning. The payment provider upgraded their API over the
weekend. They added a new field and renamed another. Your pipeline
ran on Sunday night, loaded the new data, and everything looked fine.

Except it wasn't fine. The data was garbage. And nobody noticed until
a customer called to ask why their statement showed someone else's
transactions.

Schemas change. That's not the problem. The problem is code that
assumes they won't.

## Setup

In [ ]:
%pip install -q pyarrow

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

## Two versions of the same data

We have two Parquet files from our payment provider. The first is the
original schema that our pipeline was built for. The second is the new
schema they switched to over the weekend.

Let's load both and compare.

In [ ]:
v1 = pd.read_parquet("../data/payments_v1.parquet")
print(f"v1: {v1.shape[0]} rows, {v1.shape[1]} columns")
print(f"Columns: {list(v1.columns)}")
v1.head(3)

In [ ]:
v2 = pd.read_parquet("../data/payments_v2.parquet")
print(f"v2: {v2.shape[0]} rows, {v2.shape[1]} columns")
print(f"Columns: {list(v2.columns)}")
v2.head(3)

Spot the differences:

- `sender_name` has been renamed to `payer_name`
- `reference_number` is a new column, inserted between `date` and `amount`
- `processing_fee` is a new column at the end

Two of these are additive changes (new columns). One is a breaking
change (the rename). Let's see how each type causes problems.

## The positional access trap

Here's a pattern you see in quick-and-dirty data pipelines: accessing
columns by position instead of by name.

In [ ]:
# "Column 2 is always the amount" -- famous last words
print("v1 column at index 2:", v1.columns[2])
print("v2 column at index 2:", v2.columns[2])

In v1, column index 2 is `amount`. In v2, the provider inserted
`reference_number` at position 2, so `amount` shifted to index 3.

If your code reads column 2 and treats it as the amount, it's now
reading reference numbers instead. The data loads without error. No
exception, no warning. Just wrong data flowing through the system.

Let's see this in action.

In [ ]:
def get_total_amount_by_position(df):
    """A brittle function that assumes column 2 is amount."""
    return df.iloc[:, 2].sum()


print(f"v1 total (correct):   {get_total_amount_by_position(v1):,.2f}")

# v2 has reference_number at position 2 -- this will either fail
# or produce nonsense, depending on the data type
try:
    result = get_total_amount_by_position(v2)
    print(f"v2 total (WRONG):     {result}")
except TypeError as e:
    print(f"v2 total: TypeError -- {e}")

If the new column happened to be numeric, you'd get a number back --
just the wrong number. That's worse than an error, because at least
an error gets noticed.

**Never access columns by position.** Always use column names.

## Defensive column access by name

The fix is straightforward: always refer to columns by name.

In [ ]:
def get_total_amount_by_name(df):
    """Access the amount column by name -- survives column reordering."""
    return df["amount"].sum()


print(f"v1 total: {get_total_amount_by_name(v1):,.2f}")
print(f"v2 total: {get_total_amount_by_name(v2):,.2f}")

Both return the correct total, because `amount` is still called
`amount` in both versions. The new column insertion doesn't affect
anything.

But what about the renamed column? `sender_name` became `payer_name`.
If our code references `sender_name`, it will break on v2.

In [ ]:
# This works on v1...
print("v1 senders:", v1["sender_name"].head(3).tolist())

# ...but fails on v2
try:
    print("v2 senders:", v2["sender_name"].head(3).tolist())
except KeyError as e:
    print(f"v2 KeyError: {e}")

At least a rename produces an error rather than silent corruption.
But a KeyError crashing your overnight pipeline at 3am is not exactly
ideal either. We need a better approach.

## Handling missing and renamed columns

A defensive reader should check what columns are actually present
before trying to use them. Here's one approach: check for the
column you expect, and fall back to an alternative if it's missing.

In [ ]:
def get_sender_name(df):
    """Get the sender/payer name, handling the v1->v2 rename."""
    if "sender_name" in df.columns:
        return df["sender_name"]
    elif "payer_name" in df.columns:
        return df["payer_name"]
    else:
        raise KeyError("Neither 'sender_name' nor 'payer_name' found in columns")


print("v1:", get_sender_name(v1).head(3).tolist())
print("v2:", get_sender_name(v2).head(3).tolist())

This works, but it's ad-hoc. If you have dozens of column renames
across multiple schema versions, these if/elif chains become
unmaintainable. We need something more structured.

## Inspecting Parquet schemas

Parquet files carry their schema as metadata. You can inspect it
without loading the data, which is useful for checking compatibility
before you commit to processing a file.

In [ ]:
schema_v1 = pq.read_schema("../data/payments_v1.parquet")
schema_v2 = pq.read_schema("../data/payments_v2.parquet")

print("=== v1 schema ===")
print(schema_v1)
print("\n=== v2 schema ===")
print(schema_v2)

Now let's compare them programmatically. Which columns are in both,
which are only in v1, and which are only in v2?

In [ ]:
v1_cols = set(schema_v1.names)
v2_cols = set(schema_v2.names)

print(f"In both:    {sorted(v1_cols & v2_cols)}")
print(f"Only in v1: {sorted(v1_cols - v2_cols)}")
print(f"Only in v2: {sorted(v2_cols - v1_cols)}")

This tells us exactly what changed:

- `sender_name` exists only in v1 (it was removed/renamed)
- `payer_name`, `reference_number`, and `processing_fee` exist only in v2
  (they were added, and `payer_name` is the rename of `sender_name`)

This kind of automated schema comparison is something you'd run as
part of your ingestion pipeline, before loading any data.

## Schema merging strategies

When you need to combine data from different schema versions (e.g.
historical v1 data with new v2 data), you have several options.

### Strategy 1: Intersection (only common columns)

The safest approach. You only keep columns that exist in both schemas.
You lose data, but you don't get errors.

In [ ]:
common_cols = sorted(v1_cols & v2_cols)
print(f"Common columns: {common_cols}")

combined_intersection = pd.concat(
    [v1[common_cols], v2[common_cols]],
    ignore_index=True,
)
print(f"\nCombined: {combined_intersection.shape[0]} rows, {combined_intersection.shape[1]} columns")
combined_intersection.head(3)

We lost `sender_name`/`payer_name`, `reference_number`, and
`processing_fee`. But the data is consistent and correct.

### Strategy 2: Union (all columns, with nulls)

Keep every column from every version. Where a version doesn't have a
column, fill with null.

In [ ]:
combined_union = pd.concat([v1, v2], ignore_index=True)
print(f"Combined: {combined_union.shape[0]} rows, {combined_union.shape[1]} columns")
print(f"Columns: {list(combined_union.columns)}")

# Check nulls introduced by the union
null_pct = combined_union.isna().sum() / len(combined_union) * 100
print("\nNull percentages:")
print(null_pct[null_pct > 0].to_string())

The v1 rows have nulls for `payer_name`, `reference_number`, and
`processing_fee` (they didn't exist yet). The v2 rows have nulls
for `sender_name` (it was renamed). You've kept all the data, but
now you have two columns (`sender_name` and `payer_name`) that
represent the same thing.

### Strategy 3: Explicit mapping (the right answer)

Define a mapping from each schema version to a canonical set of
column names. This gives you full control and makes the
transformation explicit and auditable.

In [ ]:
CANONICAL_COLUMNS = [
    "payment_id", "date", "amount", "currency",
    "sender_name", "recipient_name", "status",
]

COLUMN_MAPPINGS = {
    "v1": {
        "payment_id": "payment_id",
        "date": "date",
        "amount": "amount",
        "currency": "currency",
        "sender_name": "sender_name",
        "recipient_name": "recipient_name",
        "status": "status",
    },
    "v2": {
        "payment_id": "payment_id",
        "date": "date",
        "amount": "amount",
        "currency": "currency",
        "sender_name": "payer_name",  # renamed in v2
        "recipient_name": "recipient_name",
        "status": "status",
    },
}

In [ ]:
def normalise_to_canonical(df, version):
    """Map a DataFrame from a specific schema version to canonical columns."""
    mapping = COLUMN_MAPPINGS[version]
    result = pd.DataFrame()
    for canonical_name, source_name in mapping.items():
        if source_name in df.columns:
            result[canonical_name] = df[source_name]
        else:
            result[canonical_name] = pd.NA
    return result


v1_normalised = normalise_to_canonical(v1, "v1")
v2_normalised = normalise_to_canonical(v2, "v2")

combined = pd.concat([v1_normalised, v2_normalised], ignore_index=True)
print(f"Combined: {combined.shape[0]} rows, {combined.shape[1]} columns")
print(f"Columns: {list(combined.columns)}")
print(f"Nulls: {combined.isna().sum().sum()}")
combined.head(3)

No nulls. No duplicate columns. The rename is handled explicitly, and
the mapping is documented in code that anyone can read.

You do lose the v2-only columns (`reference_number`, `processing_fee`)
because they're not in the canonical schema yet. When you're ready to
use them, you add them to the canonical list and update the v1 mapping
to fill them with a sensible default or null.

## A schema-aware reader

Let's bring this together into a function that can read a Parquet
file, detect its schema version, and normalise it automatically.

In [ ]:
def detect_version(schema_names):
    """Detect the schema version from column names."""
    cols = set(schema_names)
    if "payer_name" in cols:
        return "v2"
    elif "sender_name" in cols:
        return "v1"
    else:
        raise ValueError(f"Unrecognised schema. Columns: {sorted(cols)}")


def read_payments(path):
    """Read a payments Parquet file and normalise to canonical schema."""
    schema = pq.read_schema(path)
    version = detect_version(schema.names)
    print(f"Detected schema version: {version}")

    df = pd.read_parquet(path)
    return normalise_to_canonical(df, version)


print("--- Reading v1 ---")
df1 = read_payments("../data/payments_v1.parquet")
print(f"Columns: {list(df1.columns)}\n")

print("--- Reading v2 ---")
df2 = read_payments("../data/payments_v2.parquet")
print(f"Columns: {list(df2.columns)}")

Both files produce identical column layouts. The pipeline doesn't
care which version it receives -- it handles both.

When the provider releases v3, you add a new entry to the column
mapping and a new condition to the version detector. The rest of
the pipeline stays unchanged.

## Additive vs breaking changes

Not all schema changes are equally dangerous. It helps to categorise
them:

| Change type | Example | Impact |
|---|---|---|
| Add column | New `processing_fee` column | Safe if you select by name |
| Remove column | Drop `legacy_id` | Breaks any code referencing it |
| Rename column | `sender_name` to `payer_name` | Breaks any code referencing old name |
| Change type | `amount` from int to float | May break or silently change behaviour |
| Reorder columns | Same columns, different order | Breaks positional access |

Additive changes (adding a new column) are almost always safe, as
long as you never rely on column position. Everything else is
potentially breaking.

The defensive strategies we've covered -- named access, schema
inspection, version detection, column mapping -- protect you against
all of these.

## Exercises

### Exercise 1

Update the `CANONICAL_COLUMNS` list and `COLUMN_MAPPINGS` dictionary to
include `reference_number` and `processing_fee`. For v1 data (which
doesn't have these columns), the canonical values should be `None`.

Verify that `normalise_to_canonical` produces the correct output for
both v1 and v2 data.

### Exercise 2

Write a function called `compare_schemas` that takes two Parquet file
paths and prints a report showing:

1. Columns present in both files
2. Columns added (in file 2 only)
3. Columns removed (in file 1 only)
4. Columns where the data type changed

**Hint:** Use `pq.read_schema()` and compare the field types as well
as the names.

### Exercise 3

Imagine v3 arrives with the following changes:

- `payer_name` is split into `payer_first_name` and `payer_last_name`
- `status` is renamed to `payment_status`
- `amount` type changes from float to a string with currency symbol
  (e.g. "150.00 GBP")

Which of these changes are additive, and which are breaking? For each
breaking change, describe how you would handle it in the column
mapping.

### Your turn

Write a `read_payments` function that handles v1, v2, and your
hypothetical v3 from Exercise 3. It should:

1. Detect the version from the schema
2. Apply the appropriate column mapping
3. Handle the `payer_name` split by concatenating first and last names
4. Parse the amount string to extract the numeric value

You won't have real v3 data, so create a small test DataFrame to
verify your function works.

## Summary

- **Never access columns by position.** Always use column names.
  Positional access silently breaks when columns are added, removed,
  or reordered.
- **Inspect schemas before loading data.** Parquet files carry schema
  metadata that you can read without loading the full dataset.
- **Categorise changes** as additive or breaking. Additive changes are
  safe; breaking changes require explicit handling.
- **Use column mappings** to normalise different schema versions to a
  canonical form. This makes your pipeline version-aware and keeps
  the transformation logic explicit.
- **Automate version detection** so your pipeline handles schema
  changes without manual intervention.

In the next notebook, we'll step up from pandas to a proper analytical
SQL engine -- DuckDB -- and see how it handles large-scale data
quality work.